# SAID — Spectral-Attention Integrated Detector
## Kaggle P100 Training Notebook

**Enable GPU P100:** Settings → Accelerator → GPU P100

**Required dataset:** Add `mdhabibourrahman/object-detection-dataset` via sidebar → + Add Data

In [ ]:
# ── Cell 1: Pull latest code from GitHub ─────────────────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path

REPO_URL  = 'https://github.com/habibour/foggy_object-detection_yolox.git'
REPO_DIR  = Path('/kaggle/working/repo')
WORK_DIR  = Path('/kaggle/working')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

shutil.copytree(str(REPO_DIR / 'said'), str(WORK_DIR / 'said'), dirs_exist_ok=True)
shutil.copy2(str(REPO_DIR / 'train.py'), str(WORK_DIR / 'train.py'))

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print('✅ Code ready')

In [ ]:
# ── Cell 2: Verify GPU + install ultralytics ──────────────────────────────────
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
!pip install -q --upgrade ultralytics
import ultralytics; print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
# Cell 3: Set dataset paths + write YAML configs
from pathlib import Path

WORK_DIR    = Path('/kaggle/working')
BASE        = Path('/kaggle/input/datasets/mdhabibourrahman/object-detection-dataset')

# Double-nested due to Kaggle upload structure
RTTS_ROOT   = BASE / 'RTTS_Ready'   / 'RTTS_Ready'
VOCFOG_ROOT = BASE / 'VOC_FOG_YOLO' / 'VOC_FOG_YOLO'

assert RTTS_ROOT.exists(),   f'Not found: {RTTS_ROOT}'
assert VOCFOG_ROOT.exists(), f'Not found: {VOCFOG_ROOT}'

for split in ['train', 'val', 'test']:
    p = RTTS_ROOT / 'images' / split
    print('RTTS   ', split, ':', len(list(p.iterdir())) if p.exists() else 'MISSING')

for split in ['train', 'val', 'test']:
    p = VOCFOG_ROOT / 'images' / split
    print('VOC-FOG', split, ':', len(list(p.iterdir())) if p.exists() else 'MISSING')

with open(WORK_DIR / 'rtts.yaml', 'w') as f:
    f.write(str(RTTS_ROOT) + '\n' )  # just to check
    f.seek(0); f.truncate()          # clear
    f.write('path: ' + str(RTTS_ROOT) + '\n')
    f.write('train: images/train\n')
    f.write('val: images/val\n')
    f.write('test: images/test\n')
    f.write('\nnames:\n')
    f.write('  0: person\n')
    f.write('  1: bicycle\n')
    f.write('  2: car\n')
    f.write('  3: bus\n')
    f.write('  4: motorbike\n')

with open(WORK_DIR / 'vocfog.yaml', 'w') as f:
    f.write('path: ' + str(VOCFOG_ROOT) + '\n')
    f.write('train: images/train\n')
    f.write('val: images/val\n')
    f.write('test: images/test\n')
    f.write('\nnames:\n')
    f.write('  0: person\n')
    f.write('  1: bicycle\n')
    f.write('  2: car\n')
    f.write('  3: bus\n')
    f.write('  4: motorbike\n')

print('rtts.yaml:')
print(open(WORK_DIR / 'rtts.yaml').read())

In [ ]:
# ── Cell 4: Sanity check custom modules ──────────────────────────────────────
import torch, sys
from said.a2c2f_fsa import A2C2f_FSA
from said.wiou_loss import WIoU_v3_InnerMPDIoU

m   = A2C2f_FSA(in_channels=64, out_channels=64, n_blocks=2, use_dsa=True)
x   = torch.randn(2, 64, 80, 80)
out = m(x)
assert out.shape == (2, 64, 80, 80)
print(f'✅ A2C2f_FSA: OK')

loss_fn = WIoU_v3_InnerMPDIoU(alpha=0.6, inner_scale=0.7)
pred    = torch.tensor([[10., 10., 50., 50.], [20., 20., 80., 80.]])
target  = torch.tensor([[12., 12., 52., 52.], [15., 15., 75., 75.]])
loss_fn.train()
loss = loss_fn(pred, target)
assert loss.item() >= 0
print(f'✅ WIoU_v3_InnerMPDIoU: OK')
print('\nReady to train!')

In [ ]:
# ── Cell 5: Stage 1 — Pre-train on VOC-FOG (50 epochs) ───────────────────────
!pip install -q ultralytics
!cd /kaggle/working && python train.py \
    --stage vocfog \
    --s1-epochs 50 \
    --batch 32 \
    --device 0 \
    --kaggle

In [ ]:
# ── Cell 6: Stage 2 — Fine-tune on RTTS (100 epochs) ─────────────────────────
!pip install -q ultralytics
!cd /kaggle/working && python train.py \
    --stage rtts \
    --epochs 100 \
    --batch 32 \
    --device 0 \
    --val-freq 10 \
    --test-freq 20 \
    --save-freq 5 \
    --kaggle

In [ ]:
# ── Cell 7: Final evaluation ──────────────────────────────────────────────────
!pip install -q ultralytics
!cd /kaggle/working && python train.py \
    --stage validate \
    --batch 32 \
    --device 0 \
    --kaggle

In [ ]:
# ── Cell 8: List output files for download ────────────────────────────────────
from pathlib import Path
import json

work = Path('/kaggle/working')
print('=== Saved Weights ===')
for f in sorted(work.rglob('*.pt')):
    print(f'  {f.relative_to(work)}  ({f.stat().st_size/1e6:.1f} MB)')

print('\n=== Eval Log ===')
log = work / 'runs/said/stage2b_full/eval_log.csv'
if log.exists():
    print(log.read_text())

print('\n=== Summary ===')
summary = work / 'runs/said/stage2b_full/said_summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))